## Brain Map Visualization - Subcortical Volume (aseg)

### Overview

Subcortical-only companion to `ggseg_volume_desikanaseg.ipynb`. Visualizes **Hedges' g**
for 16 subcortical structures using the **FreeSurfer aseg atlas** in `ggseg` 2.x.

**Subcortical structures:** Thalamus · Caudate · Putamen · Pallidum · Hippocampus ·
Amygdala · Accumbens area · Ventral DC (8 structures × 2 hemispheres = 16 parcels)

FDR correction is applied **only within the 16 subcortical tests** per contrast (less
conservative than correcting jointly with 68 cortical parcels).

**Reference:** Mowinckel & Vidal-Piñeiro (2020). *Advances in Methods and Practices in
Psychological Science*.

In [ ]:
# install.packages(c("tidyverse", "patchwork", "remotes"))
# remotes::install_github("LCBC-UiO/ggseg")
suppressPackageStartupMessages({
  library(ggseg); library(tidyverse); library(patchwork)
})
cat("ggseg:", as.character(packageVersion("ggseg")), "\n")

### 1. Load pre-computed T-statistics

Results come from `volume_subcortical_shi.ipynb` (Python), which ran covariate-adjusted
OLS group comparisons (covariates: age, SEX, eTIV, field strength) and saved one CSV
per contrast. This avoids running uncorrected Welch's t-tests on raw volumes here.

**Sign convention:** In the Python OLS model, `group01 = {g1: 0, g2: 1}`, so a positive
T-value means g2 has more volume. After negating below, **positive Hedges' g = more
volume in g1** (the first-named group, e.g. De Novo PD in "De Novo PD vs HC").

In [ ]:
RESULTS_DIR <- "../../results"
FIG_DIR     <- file.path(RESULTS_DIR, "figures", "ggseg_volume_subcortical")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

# Group sizes matching the Python analysis
GROUP_N <- list(
  "De Novo PD vs HC"           = c(n1 = 558, n2 = 192),
  "Prodromal PD vs HC"         = c(n1 = 279, n2 = 192),
  "De Novo PD vs Prodromal PD" = c(n1 = 558, n2 = 279)
)

# Pre-computed OLS T-statistics from volume_subcortical_shi.ipynb
TTEST_FILES <- list(
  "De Novo PD vs HC"           = file.path(RESULTS_DIR, "de_novo_pd_vs_hc_parcelwise_ttest_volume_subcortical.csv"),
  "Prodromal PD vs HC"         = file.path(RESULTS_DIR, "prodromal_pd_vs_hc_parcelwise_ttest_volume_subcortical.csv"),
  "De Novo PD vs Prodromal PD" = file.path(RESULTS_DIR, "de_novo_pd_vs_prodromal_pd_parcelwise_ttest_volume_subcortical.csv")
)

df_sc <- imap_dfr(TTEST_FILES, function(path, cname) {
  read_csv(path, show_col_types = FALSE) %>% mutate(contrast = cname)
}) %>%
  filter(!is.na(pvalue))   # remove VentralDC rows (all-NaN in Python)

cat("Loaded", nrow(df_sc), "parcel x contrast rows\n")
cat("Parcels per contrast:", n_distinct(df_sc$parcel), "\n")
cat("T-value range:", round(range(df_sc$Tvalue, na.rm = TRUE), 3), "\n")

### 2. Compute Hedges' g from OLS T-statistic

Hedges' g is derived from the OLS T-statistic saved in the CSV:

$$g = -T \cdot \sqrt{\frac{1}{n_1} + \frac{1}{n_2}} \cdot \left(1 - \frac{3}{4 \cdot df_{resid} - 1}\right)$$

The **negation** aligns the sign convention with the contrast name order:
positive g = more volume in the **first-named group** (g1, e.g. De Novo PD).

This matches the formula used in `volume_subcortical_shi.ipynb` (Python), where the
OLS T-statistic was defined as the effect of moving from g1 → g2, so positive T = g2 > g1.

In [ ]:
contrasts <- list(
  "De Novo PD vs HC"           = c(1, 2),
  "Prodromal PD vs HC"         = c(4, 2),
  "De Novo PD vs Prodromal PD" = c(1, 4)
)

# Hedges' g from OLS T-statistic
# Negated so that positive g = more volume in g1 (first-named group)
hedges_g_from_t <- function(t, n1, n2, df_resid) {
  -t * sqrt(1/n1 + 1/n2) * (1 - 3 / (4 * df_resid - 1))
}

df_sc <- df_sc %>%
  rowwise() %>%
  mutate(g = hedges_g_from_t(
    Tvalue,
    GROUP_N[[contrast]][["n1"]],
    GROUP_N[[contrast]][["n2"]],
    df
  )) %>%
  ungroup()

cat("Hedges' g range:", round(range(df_sc$g, na.rm = TRUE), 3), "\n")

### 3. Build aseg label column

The aseg atlas in ggseg 2.x joins on a `label` column with the format:
`"Left-Hippocampus"`, `"Right-Thalamus"`, `"Left-Accumbens-area"`, …

Key mapping: `Accumbens+area` → `Accumbens-area` (replace `+` with `-`).

Only the `label` + fill variable are passed to `ggplot()`. Extra columns sharing names
with atlas columns (`hemi`, `region`, …) break the automatic join.

In [ ]:
df_sc <- df_sc %>%
  mutate(
    hemi      = if_else(str_detect(parcel, "hemi-L"), "left", "right"),
    raw_label = str_replace(parcel, ".*_lab-", ""),
    # aseg label: "Left-Hippocampus", "Right-Accumbens-area", etc.
    label     = paste0(
      if_else(hemi == "left", "Left", "Right"), "-",
      str_replace(raw_label, "\\+", "-")
    ),
    region    = str_replace(raw_label, "\\+area", " area")
  )

# Verify against atlas
atlas_labels <- as.data.frame(aseg())$label
bad <- df_sc %>% filter(!label %in% atlas_labels) %>% pull(label) %>% unique()
if (length(bad) == 0) {
  cat("All", n_distinct(df_sc$label), "labels match the aseg atlas.\n")
} else {
  cat("WARNING - unmatched labels:", bad, "\n")
}

### 4. FDR correction

Benjamini-Hochberg FDR applied within each contrast across the 16 subcortical tests only.

In [ ]:
df_sc <- df_sc %>%
  group_by(contrast) %>%
  mutate(p_fdr = p.adjust(pvalue, method="fdr")) %>%
  ungroup()

df_sc %>%
  group_by(contrast) %>%
  summarise(
    n_sig_fdr = sum(p_fdr < 0.05, na.rm=TRUE),
    n_sig_unc = sum(pvalue < 0.05, na.rm=TRUE),
    n         = n(),
    g_max_abs = round(max(abs(g), na.rm=TRUE), 3)
  )

### 5. Colour scale

Blue–white–red diverging scale, limits ±0.8 (large effect threshold).
Subcortical structures in PD can show larger volume differences than cortical parcels.

In [ ]:
G_LIMIT <- 0.8

scale_g <- scale_fill_gradient2(
  low="#2166AC", mid="white", high="#D6604D",
  midpoint=0, limits=c(-G_LIMIT, G_LIMIT),
  oob=scales::squish, name="Hedges' g", na.value="grey85"
)

### 6. Subcortical brain maps (coronal view)

For each contrast: full Hedges' g map (left) + FDR-masked map (right).
Only the **coronal slice** (`view == "coronal_1"`) is shown — a single horizontal cross-section
through the brain that captures all seven bilateral subcortical structure pairs simultaneously.

**Why coronal only?** The aseg atlas contains 7 views (axial × 4, coronal × 2, sagittal).
For subcortical structures, the coronal slice is the most informative single view and matches
the style used in published ggseg subcortical figures.

`brain_join()` flattens the atlas into a filterable `sf` data frame; `geom_sf()` renders it.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 3.5)

# Helper: join data to aseg atlas and filter to coronal slice
coronal_join <- function(df) {
  brain_join(df, aseg()) %>% dplyr::filter(view == "coronal_1")
}

for (cname in names(contrasts)) {
  d <- df_sc %>% filter(contrast == cname)

  p_full <- ggplot(coronal_join(d %>% select(label, g))) +
    geom_sf(aes(fill = g), colour = "white") +
    scale_g +
    labs(
      title    = cname,
      subtitle = "Subcortical volume - Hedges' g (all structures; eTIV-corrected)"
    ) +
    theme_void() +
    theme(
      plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
      legend.position = "right"
    )

  p_mask <- ggplot(
    coronal_join(d %>% mutate(g_sig = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_sig))
  ) +
    geom_sf(aes(fill = g_sig), colour = "white") +
    scale_g +
    labs(
      title    = cname,
      subtitle = "Subcortical volume - FDR-masked (q < 0.05; eTIV-corrected)"
    ) +
    theme_void() +
    theme(
      plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
      legend.position = "right"
    )

  combined <- p_full + p_mask + plot_layout(guides = "collect") & theme(legend.position = "right")
  print(combined)
  fname <- tolower(str_replace_all(cname, " ", "_"))
  ggsave(file.path(FIG_DIR, paste0(fname, ".png")), combined, width = 10, height = 3.5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 6b. Paper-style brain maps (sequential red scale, FDR-only)

Replicating the style of Laansma et al. (following the figure caption):
> *"Cohen's d values were calculated and are presented in the figure when the FDR-adjusted
> p value reached < 0.05. Darker red indicates more atrophy."*

**Design choices:**
- **Sequential white → dark red** scale: encodes *magnitude* of atrophy, not direction
- **Absolute |Hedges' g|** displayed — only for FDR-significant parcels
- **Non-significant parcels** → very light grey (`#f5f5f5`) so the brain outline stays visible
- **Grey parcel borders** (`grey60`) make individual parcels distinguishable
- One panel per contrast

In [ ]:
FIG_DIR_PAPER <- file.path(FIG_DIR, "paper_style")
dir.create(FIG_DIR_PAPER, recursive = TRUE, showWarnings = FALSE)

G_LIMIT_PAPER <- ceiling(max(abs(df_sc$g), na.rm = TRUE) * 10) / 10

scale_g_paper <- scale_fill_gradient(
  low      = "white",
  high     = "#4393C3",
  limits   = c(0, G_LIMIT_PAPER),
  oob      = scales::squish,
  name     = "|Hedges' g|",
  na.value = "#f5f5f5"
)

cat("Paper scale upper limit:", G_LIMIT_PAPER, "\n")
options(repr.plot.width = 6, repr.plot.height = 3.5)

for (cname in names(contrasts)) {

  n_sig <- df_sc %>% filter(contrast == cname, p_fdr < 0.05) %>% nrow()

  bj <- coronal_join(
    df_sc %>%
      filter(contrast == cname) %>%
      mutate(g_abs = if_else(p_fdr < 0.05, abs(g), NA_real_)) %>%
      select(label, g_abs)
  )

  p_paper <- ggplot(bj) +
    geom_sf(aes(fill = g_abs), colour = "grey60") +
    scale_g_paper +
    labs(
      title    = cname,
      subtitle = paste0("Subcortical volume - |Hedges' g| (FDR q < 0.05 only; n_sig = ", n_sig, ")")
    ) +
    theme_void() +
    theme(
      plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
      legend.position = "right"
    )

  print(p_paper)

  fname <- paste0("paper_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(
    filename = file.path(FIG_DIR_PAPER, paste0(fname, ".png")),
    plot     = p_paper,
    width    = 6, height = 3.5, dpi = 300
  )
  cat("Saved:", fname, "(", n_sig, "significant structures)\n")
}

### Figure 6b — Thesis caption (recommended primary subcortical figure)

**Figure 6b.** FDR-corrected subcortical volume differences across three contrasts: (a) De Novo PD vs HC, (b) Prodromal PD vs HC, (c) De Novo PD vs Prodromal PD. Only structures reaching FDR-corrected significance (Benjamini-Hochberg q < 0.05) are coloured; colour intensity encodes effect magnitude (|Hedges' g|), scaled 0–0.30. Grey structures did not reach FDR correction. Coronal slice from the FreeSurfer Aseg atlas. OLS T-statistics adjusted for age, sex, estimated total intracranial volume (eTIV), and MRI field strength across 14 bilateral structures.

The only FDR-significant subcortical finding was bilateral pallidal enlargement in De Novo PD relative to Prodromal PD (left pallidum: g=+0.205, T=−2.795, q=0.043; right pallidum: g=+0.201, T=−2.749, q=0.043), visible in panel (c). Positive Hedges' g indicates greater volume in De Novo PD. Neither De Novo PD nor Prodromal PD differed significantly from HC in any subcortical structure (panels a–b). The pallidal enlargement in manifest relative to prodromal PD is consistent with indirect-pathway hyperactivation accompanying the transition to overt motor dysfunction.

### 6c. Archive-style: Atrophy-only map (white → pink)

Replicating the style of `archive/subcortical_atrophy_PD_HC.R` and
`archive/combined_subcortical_atrophy_plots.R`.

**What is shown:** Only structures where the first-named group has **less** volume (g < 0 =
atrophy). Structures with g ≥ 0 (no atrophy or larger volume in first-named group) are set to
NA and appear in grey90. The value plotted is the **absolute |Hedges' g|** of atrophy.

**Colour scale:** white → "pink", limits 0–0.30, `na.value = "grey90"` — identical to the
archive script.

**Combined figure:** all 3 contrasts stacked vertically (patchwork `/`), shared legend on the
right. Output dimensions: 5 × 9 inches (matching archive).

---

**Difference from current styles:**

| Style | What is shown | Colour | Non-atrophy parcels |
|---|---|---|---|
| **Archive / this section** | g < 0 only → \|g\| (atrophy magnitude) | white → pink | grey90 |
| **Full map (section 6)** | all g (both directions) | blue–white–red (diverging) | coloured |
| **Paper style (section 6b)** | \|g\| for FDR-sig only | white → dark red | #f5f5f5 |

The archive style is publication-ready for showing *where* atrophy occurs; the full map is
better for exploratory analysis; the paper style (Laansma et al.) is the strictest (FDR gate).

In [ ]:
FIG_DIR_ATROPHY <- file.path(FIG_DIR, "atrophy_style")
dir.create(FIG_DIR_ATROPHY, recursive = TRUE, showWarnings = FALSE)

scale_atrophy <- scale_fill_gradient(
  low      = "white",
  high     = "pink",
  limits   = c(0, 0.30),
  oob      = scales::squish,
  na.value = "grey90",
  name     = "Atrophy\n(|Hedges' g|)"
)

contrast_desc_sc <- c(
  "De Novo PD vs HC"           = "De Novo PD shows less volume than HC",
  "Prodromal PD vs HC"         = "Prodromal PD shows less volume than HC",
  "De Novo PD vs Prodromal PD" = "De Novo PD shows less volume than Prodromal PD"
)

options(repr.plot.width = 7, repr.plot.height = 4.5)
plots_atrophy <- list()

for (cname in names(contrasts)) {
  bj <- coronal_join(
    df_sc %>%
      filter(contrast == cname) %>%
      mutate(atrophy = if_else(g < 0, abs(g), NA_real_)) %>%
      select(label, atrophy)
  )
  n_unc <- sum(df_sc$contrast == cname & df_sc$pvalue < 0.05 & df_sc$g < 0, na.rm = TRUE)

  p <- ggplot(bj) +
    geom_sf(aes(fill = atrophy), colour = "grey70") +
    scale_atrophy +
    labs(
      title    = cname,
      subtitle = paste0(contrast_desc_sc[[cname]], " (pink)  |  ", n_unc, " structures p<0.05 uncorrected")
    ) +
    theme_void() +
    theme(
      plot.title      = element_text(hjust = 0.5, size = 10, face = "bold",
                                     margin = margin(t = 10, b = 3)),
      plot.subtitle   = element_text(hjust = 0.5, size = 8, colour = "grey35",
                                     margin = margin(b = 10)),
      legend.position = "right",
      plot.margin     = margin(t = 8, r = 8, b = 12, l = 8)
    )

  plots_atrophy[[cname]] <- p
  print(p)
  fname <- paste0("atrophy_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(file.path(FIG_DIR_ATROPHY, paste0(fname, ".png")), p, width = 7, height = 4.5, dpi = 300)
  cat("Saved:", fname, "\n")
}

combined_atrophy <- (plots_atrophy[[1]] / plots_atrophy[[2]] / plots_atrophy[[3]]) +
  plot_layout(guides = "collect") &
  theme(legend.position = "right")

options(repr.plot.width = 7, repr.plot.height = 14)
print(combined_atrophy)

out_combined <- file.path(FIG_DIR_ATROPHY, "combined_subcortical_atrophy_maps.png")
ggsave(out_combined, combined_atrophy, width = 7, height = 14, dpi = 300)
cat("Saved combined:", out_combined, "\n")

### 6d. Combined atrophy figure: unthresholded vs FDR-corrected (labelled a–f)

Publication-ready 3 × 2 figure combining all contrasts and both thresholding approaches.

**Layout:**

| | Left column | Right column |
|---|---|---|
| **Row 1 (a, b)** | De Novo PD vs HC — all volume loss | De Novo PD vs HC — FDR q < 0.05 |
| **Row 2 (c, d)** | Prodromal PD vs HC — all volume loss | Prodromal PD vs HC — FDR q < 0.05 |
| **Row 3 (e, f)** | De Novo PD vs Prodromal PD — all volume loss | De Novo PD vs Prodromal PD — FDR q < 0.05 |

**Left panels (a, c, e):** All subcortical structures with g < 0 (volume loss in first-named
group), unthresholded. Shows the full effect landscape.

**Right panels (b, d, f):** FDR-significant volume loss only (g < 0 and q < 0.05). Only De
Novo PD vs Prodromal PD has surviving structures (bilateral pallidum, q = 0.043).

Scale: white → pink, |Hedges' g|, limits 0–0.30.
Panel letters (a–f) added via patchwork `tag_levels`.

In [ ]:
FIG_DIR_COMBINED <- file.path(FIG_DIR, "combined_atrophy")
dir.create(FIG_DIR_COMBINED, recursive = TRUE, showWarnings = FALSE)

scale_pink <- scale_fill_gradient(
  low      = "white",
  high     = "pink",
  limits   = c(0, 0.30),
  oob      = scales::squish,
  na.value = "grey90",
  name     = "Atrophy\n(|Hedges' g|)"
)

contrast_titles <- c(
  "De Novo PD vs HC"           = "De Novo PD vs HC",
  "Prodromal PD vs HC"         = "Prodromal PD vs HC",
  "De Novo PD vs Prodromal PD" = "De Novo PD vs Prodromal PD"
)

panel_theme <- theme(
  plot.title    = element_text(hjust = 0.5, size = 9.5, face = "bold",
                               margin = margin(t = 10, b = 3)),
  plot.subtitle = element_text(hjust = 0.5, size = 7.5, colour = "grey35",
                               margin = margin(b = 10)),
  plot.margin   = margin(t = 8, r = 6, b = 12, l = 6)
)

plots_unthresh <- list()
plots_fdr_only <- list()

for (cname in names(contrasts)) {
  d     <- df_sc %>% filter(contrast == cname)
  n_sig <- sum(d$p_fdr < 0.05 & d$g < 0, na.rm = TRUE)
  sig_label <- if (n_sig == 0) "none" else {
    d %>% filter(p_fdr < 0.05 & g < 0) %>% pull(region) %>% unique() %>% paste(collapse = ", ")
  }

  plots_unthresh[[cname]] <- ggplot(
    coronal_join(d %>% mutate(atrophy = if_else(g < 0, abs(g), NA_real_)) %>% select(label, atrophy))
  ) +
    geom_sf(aes(fill = atrophy), colour = "grey70") +
    scale_pink +
    labs(
      title    = contrast_titles[[cname]],
      subtitle = paste0(contrast_desc_sc[[cname]], " (pink)  |  unthresholded")
    ) +
    theme_void() + panel_theme

  plots_fdr_only[[cname]] <- ggplot(
    coronal_join(d %>% mutate(atrophy = if_else(g < 0 & p_fdr < 0.05, abs(g), NA_real_)) %>% select(label, atrophy))
  ) +
    geom_sf(aes(fill = atrophy), colour = "grey70") +
    scale_pink +
    labs(
      title    = contrast_titles[[cname]],
      subtitle = paste0("FDR q<0.05: n=", n_sig, " (", sig_label, ")")
    ) +
    theme_void() + panel_theme
}

fig_combined <- (
  (plots_unthresh[[1]] | plots_fdr_only[[1]]) /
  (plots_unthresh[[2]] | plots_fdr_only[[2]]) /
  (plots_unthresh[[3]] | plots_fdr_only[[3]])
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title      = "Subcortical Volume: Unthresholded vs FDR-corrected Volume Loss",
    subtitle   = "Left (a, c, e): all volume loss (g < 0) | Right (b, d, f): FDR-corrected (q < 0.05)",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

options(repr.plot.width = 10, repr.plot.height = 14)
print(fig_combined)

out_path <- file.path(FIG_DIR_COMBINED, "combined_subcortical_volume_unthresh_vs_fdr.png")
ggsave(out_path, fig_combined, width = 10, height = 14, dpi = 300)
cat("Saved:", out_path, "\n")

### Figure 6d — Thesis caption

**Figure 6d-1 (unthresholded).** Subcortical volume loss across three group contrasts: (a) De Novo PD vs HC, (b) Prodromal PD vs HC, (c) De Novo PD vs Prodromal PD. Pink shading indicates subcortical structures where the first-named group has reduced volume relative to the reference group (Hedges' g < 0); colour intensity encodes effect magnitude (|Hedges' g|), scaled 0–0.30. Grey structures show no volume loss or greater volume in the first-named group. Coronal slice from the FreeSurfer Aseg atlas. OLS T-statistics adjusted for age, sex, estimated total intracranial volume (eTIV), and MRI field strength.

**Figure 6d-2 (FDR-corrected).** Same contrasts as 6d-1, retaining only structures that show FDR-corrected volume loss in the first-named group (g < 0, q < 0.05). All panels appear grey: no subcortical structure exhibits FDR-significant volume loss in any contrast. Note that the only FDR-significant subcortical finding — bilateral pallidal enlargement in De Novo PD relative to Prodromal PD (left: g=+0.205, q=0.043; right: g=+0.201, q=0.043) — is in the opposite direction (greater volume in De Novo PD) and is therefore not shown in this atrophy-direction figure. See Figure 6b for the FDR-corrected map showing this effect.

### 6e. Combined paper-style figure: unthresholded vs FDR-corrected

Publication-ready 3 × 2 figure equivalent to Figure 6d, but using the paper-style scale
(white → light blue, |Hedges' g|) instead of the atrophy-direction pink scale.

**Layout:**

| | Left column | Right column |
|---|---|---|
| **Row 1 (a, b)** | De Novo PD vs HC — all \|g\| unthresholded | De Novo PD vs HC — FDR q < 0.05 |
| **Row 2 (c, d)** | Prodromal PD vs HC — all \|g\| | Prodromal PD vs HC — FDR q < 0.05 |
| **Row 3 (e, f)** | De Novo PD vs Prodromal PD — all \|g\| | De Novo PD vs Prodromal PD — FDR q < 0.05 |

**Key difference from Figure 6d:** The paper-style scale shows |Hedges' g| for all structures
regardless of direction (not just volume-loss g < 0), so the bilateral pallidum (De Novo PD
has *greater* volume than Prodromal PD, g ≈ +0.20, q = 0.043) correctly appears in
panel (f). Panel (b) and (d) remain grey (no FDR-significant result vs HC).

**Colour scale:** white → light blue (#4393C3), |Hedges' g|, shared with Figure 6b.

In [ ]:
plots_pap_unc <- list()
plots_pap_fdr <- list()

for (cname in names(contrasts)) {
  d     <- df_sc %>% filter(contrast == cname)
  n_sig <- sum(d$p_fdr < 0.05, na.rm = TRUE)
  sig_label <- if (n_sig == 0) "none" else {
    d %>% filter(p_fdr < 0.05) %>% pull(region) %>% unique() %>% paste(collapse = ", ")
  }

  plots_pap_unc[[cname]] <- ggplot(
    coronal_join(d %>% mutate(g_abs = abs(g)) %>% select(label, g_abs))
  ) +
    geom_sf(aes(fill = g_abs), colour = "grey60") +
    scale_g_paper +
    labs(
      title    = contrast_titles[[cname]],
      subtitle = "|Hedges' g| (all structures, unthresholded)"
    ) +
    theme_void() + panel_theme

  plots_pap_fdr[[cname]] <- ggplot(
    coronal_join(
      d %>% mutate(g_abs = if_else(p_fdr < 0.05, abs(g), NA_real_)) %>% select(label, g_abs)
    )
  ) +
    geom_sf(aes(fill = g_abs), colour = "grey60") +
    scale_g_paper +
    labs(
      title    = contrast_titles[[cname]],
      subtitle = paste0("FDR q<0.05: n=", n_sig, " (", sig_label, ")")
    ) +
    theme_void() + panel_theme
}

fig_paper_combined <- (
  (plots_pap_unc[[1]] | plots_pap_fdr[[1]]) /
  (plots_pap_unc[[2]] | plots_pap_fdr[[2]]) /
  (plots_pap_unc[[3]] | plots_pap_fdr[[3]])
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title      = "Subcortical Volume (Unthresholded vs FDR-corrected)",
    subtitle   = "Left (a, c, e): all |Hedges' g| | Right (b, d, f): FDR-corrected (q < 0.05)",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

options(repr.plot.width = 10, repr.plot.height = 14)
print(fig_paper_combined)

out_path <- file.path(FIG_DIR_PAPER, "combined_paper_style_unthresh_vs_fdr.png")
ggsave(out_path, fig_paper_combined, width = 10, height = 14, dpi = 300)
cat("Saved:", out_path, "\n")

### 6g. Diverging pastel maps: both directions (light pink / light blue)

Publication-ready figure showing **both positive and negative Hedges' g simultaneously**
using a pastel diverging colour scale that matches the study's existing palette:

- **Light pink** (`lightpink`): negative g — first-named group has *less* volume
- **Light blue** (`#AED6F1`): positive g — first-named group has *more* volume
- **White**: no difference (g = 0)
- **Grey85**: no data

Unlike the atrophy-only (section 6c) figures, this figure shows the full landscape of
subcortical volume differences, making the pallidal enlargement in De Novo PD vs Prodromal
PD (g ≈ +0.20, FDR q = 0.043) visible in blue alongside any volume-loss structures in pink.

Scale limits: ±0.30, encompassing the full observed range (|g|_max ≈ 0.205).
All 14 bilateral structures are coloured; significance indicated in subtitle only.

In [ ]:
CNAMES_DISPLAY_SC <- c(
  "De Novo PD vs HC"           = "de novo PD vs HC",
  "Prodromal PD vs HC"         = "prodromal PD vs HC",
  "De Novo PD vs Prodromal PD" = "de novo PD vs prodromal PD"
)

G_LIMIT_PASTEL <- 0.30

scale_diverging_pastel <- scale_fill_gradient2(
  low      = "lightpink",
  mid      = "white",
  high     = "#AED6F1",
  midpoint = 0,
  limits   = c(-G_LIMIT_PASTEL, G_LIMIT_PASTEL),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

plots_div_fdr <- list()
plots_div_unc <- list()

for (cname in names(contrasts)) {
  d     <- df_sc %>% filter(contrast == cname)

  plots_div_fdr[[cname]] <- ggplot(
    coronal_join(
      d %>% mutate(g_plot = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_plot)
    )
  ) +
    geom_sf(aes(fill = g_plot), colour = "grey60") +
    scale_diverging_pastel +
    labs(title = CNAMES_DISPLAY_SC[[cname]]) +
    theme_void() + panel_theme

  plots_div_unc[[cname]] <- ggplot(
    coronal_join(d %>% mutate(g_plot = g) %>% select(label, g_plot))
  ) +
    geom_sf(aes(fill = g_plot), colour = "grey60") +
    scale_diverging_pastel +
    labs(
      title    = CNAMES_DISPLAY_SC[[cname]],
    ) +
    theme_void() + panel_theme
}

options(repr.plot.width = 10, repr.plot.height = 14)

fig_div_combined <- (
  (plots_div_unc[[1]] | plots_div_fdr[[1]]) /
  (plots_div_unc[[2]] | plots_div_fdr[[2]]) /
  (plots_div_unc[[3]] | plots_div_fdr[[3]])
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Subcortical Volume: Diverging Hedges' g",
    subtitle = "Left: all structures, unthresholded | Right: FDR q < 0.05 only\nLight pink = volume loss (g < 0) | Light blue = volume gain (g > 0)",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_div_combined)

out_path <- file.path(FIG_DIR, "figure_diverging_subcortical_combined.png")
ggsave(out_path, fig_div_combined, width = 10, height = 14, dpi = 300)
cat("Saved:", out_path, "\n")

### Figure 6g — Thesis caption

**Figure 6g** (3 × 2 combined). Diverging subcortical volume maps across three group contrasts. Each row shows one contrast: (a, b) *de novo* PD vs HC; (c, d) prodromal PD vs HC; (e, f) *de novo* PD vs prodromal PD. Left panels show only FDR-significant structures (Benjamini–Hochberg q < 0.05); right panels show all 14 bilateral structures, unthresholded. Light pink = less volume in first-named group (g < 0); light blue = more volume (g > 0); white = no difference; grey = not FDR-significant (left panels) or no data. Panel subtitles report FDR-significant regions (left) or nominally significant structure counts (right).

The only FDR-significant finding is bilateral pallidal enlargement in *de novo* PD vs prodromal PD (panels e–f; left pallidum: g = +0.205, q = 0.043; right pallidum: g = +0.201, q = 0.043), visible as light blue structures in panel (e). All other contrasts show no FDR-significant subcortical volume differences.

### Figure 6e — Thesis caption

**Figure 6e-1 (unthresholded).** Subcortical volume effect size maps across three group contrasts: (a) De Novo PD vs HC, (b) Prodromal PD vs HC, (c) De Novo PD vs Prodromal PD. Colour intensity (white to light blue) encodes effect magnitude (|Hedges' g|) for all subcortical structures, regardless of direction or statistical significance. Structures with larger absolute differences between groups appear in darker blue. Coronal slice from the FreeSurfer Aseg atlas. OLS T-statistics adjusted for age, sex, estimated total intracranial volume (eTIV), and MRI field strength.

**Figure 6e-2 (FDR-corrected).** Same contrasts as 6e-1, retaining only structures reaching FDR-corrected significance (Benjamini-Hochberg q < 0.05). Panels (b) and (d) appear grey, indicating no FDR-significant subcortical volume differences in comparisons against HC. Panel (f) shows bilateral pallidal enlargement in De Novo PD relative to Prodromal PD (left pallidum: g=+0.205, q=0.043; right pallidum: g=+0.201, q=0.043), the only FDR-significant subcortical finding across all contrasts. Positive Hedges' g indicates greater volume in De Novo PD.

### 7. All contrasts side by side

Three-panel figure showing Hedges' g for all contrasts simultaneously for comparison.
Useful for assessing effect size progression from prodromal to de novo PD.

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 3.5)

g_panels <- map(names(contrasts), function(cname) {
  ggplot(coronal_join(df_sc %>% filter(contrast == cname) %>% select(label, g))) +
    geom_sf(aes(fill = g), colour = "white") +
    scale_g +
    labs(title = cname, subtitle = "Hedges' g (eTIV-corrected)") +
    theme_void() +
    theme(
      plot.title      = element_text(hjust = 0.5, size = 11, face = "bold"),
      plot.subtitle   = element_text(hjust = 0.5, size = 9,  colour = "grey40"),
      legend.position = "right"
    )
})

all_panel <- g_panels[[1]] + g_panels[[2]] + g_panels[[3]] +
  plot_layout(guides = "collect", nrow = 1) & theme(legend.position = "right")
print(all_panel)
ggsave(file.path(FIG_DIR, "all_contrasts.png"), all_panel, width = 18, height = 3.5, dpi = 300)
cat("Saved: all_contrasts\n")

### 8. Results table

All subcortical structures sorted by absolute Hedges' g.
`*` = FDR significant (q < 0.05) within subcortical tests.

In [ ]:
df_sc %>%
  select(contrast, hemi, region, g, Tvalue, pvalue, p_fdr) %>%
  mutate(
    across(where(is.numeric), \(x) round(x, 4)),
    sig = case_when(p_fdr < 0.001 ~ "***", p_fdr < 0.01 ~ "**",
                    p_fdr < 0.05  ~ "*",   TRUE ~ "")
  ) %>%
  arrange(contrast, desc(abs(g)))

---
## Notes

### Data source
T-statistics and p-values come from `volume_subcortical_shi.ipynb` (Python). The OLS
models included age, SEX, eTIV, and field strength as covariates — producing properly
adjusted effect sizes rather than raw group differences.

### Hedges' g sign convention
Hedges' g is derived as `g = −T × sqrt(1/n1 + 1/n2) × J` so that:
- **Positive g (red)** = more volume in the first-named group (e.g. De Novo PD in "De Novo PD vs HC")
- **Negative g (blue)** = less volume in the first-named group (atrophy relative to reference)

### Why the results differ from the previous version
The previous notebook computed Welch's t-tests on **raw volumes** without covariate
adjustment. eTIV confounding caused some structures to appear larger in PD (positive raw g),
while the eTIV-adjusted results show the opposite. The current results are methodologically
correct and consistent with the Python analysis pipeline.

### FDR scope
FDR correction is applied across 14 subcortical tests per contrast. VentralDC is excluded
(no data in the PPMI dataset for this structure).

### Group codes
CONCOHORT 1 = De Novo PD · 2 = HC · 4 = Prodromal PD